In [ ]:
# package(s) used for creating and geo-locating the graph
import networkx as nx
import pyproj
from shapely.geometry import Point, LineString, Polygon
from shapely.ops import transform

# package(s) related to the simulation (creating the vessel, running the simulation)
import datetime
import simpy
import opentnsim
from opentnsim.core import Identifiable, Movable, VesselProperties, ExtraMetadata
from opentnsim.core.utils import create_object
from opentnsim.utils import inspect_object, generate_vessels_from_distribution
from opentnsim.core.logutils import logbook2eventtable
from opentnsim.core.visualizations import generate_vessel_gantt_chart
from opentnsim.graph.mixins import HasMultiDiGraph
from opentnsim.graph.utils import find_nodes_in_a_polygon
from opentnsim.graph.calculations import merge_two_consecutive_edges_based_on_shared_node
from opentnsim.graph.visualizations import plot_graph_folium, visualize_geometry_polygon_in_folium_plot
from opentnsim.lock.visualizations import spatially_visualize_lock_complex
from opentnsim.energy.mixins import ConsumesEnergy
from opentnsim.output import HasOutput
from scipy.stats import norm, uniform, expon
from opentnsim.graph.utils import get_closest_node_to_geometry
import folium

# import of modules important for locking
from opentnsim.lock import IsLockChamber, IsLockWaitingArea, IsLockComplex, LockComplexTraversable
import geopandas as gpd
import opentnsim.core.utils as core_utils
from opentnsim import fis

# package(s) needed for inspecting the output
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# package(s) needed for adding the hydrodynamic data
import xarray as xr
from pyproj.transformer import Transformer
from opentnsim.environment.mixins.hydrodynamics import HydrodynamicData, HydrodynamicDataManager

from pathlib import Path
import requests
print("This notebook is executed with OpenTNSim version {}".format(opentnsim.__version__))

In [ ]:
%load_ext autoreload
%autoreload 2

#### 0. Create environment

In [ ]:
# start simpy environment
simulation_start = datetime.datetime(2026, 1, 1, 0, 0, 0)
simulation_stop = datetime.datetime(2026, 2, 1, 0, 0, 0)
env = simpy.Environment(initial_time=simulation_start.timestamp())
env.simulation_start = simulation_start
env.simulation_stop = simulation_stop
env.epoch = simulation_start

#### 1. Create graph
Next we create a network (a graph) along which the vessel can move. For this case we use the Fairway Information System graph, and make the vessels sail from one side of the lock to another side.

In [ ]:
# load the processed version from the Fairway Information System graph provided by Rijkswaterstaat
FG = fis.load_network(network = "euris", version="0.1")

#### 1+ Adapt graph
We download the lock geometries of Terneuzen, and focus on the following area of interest:

In [ ]:
area_of_interest = Polygon([Point(3.8, 51.3), Point(3.8, 51.35), Point(3.85, 51.35), Point(3.85, 51.3)])

lon_min = np.min(area_of_interest.exterior.coords.xy[0])
lon_max = np.max(area_of_interest.exterior.coords.xy[0])
lat_min = np.min(area_of_interest.exterior.coords.xy[1])
lat_max = np.max(area_of_interest.exterior.coords.xy[1])

In [ ]:
# Use faster server
url = "https://overpass.kumi.systems/api/interpreter"

# Berlin bounding box (faster than area lookup)
query = f"""
[out:json][timeout:180];
nwr["water"~"lock"]({lat_min},{lon_min},{lat_max},{lon_max});
out geom;
"""

response = requests.get(url, params={'data': query})
data = response.json()

# Convert to DataFrame
elements = data["elements"]

df = pd.json_normalize(elements)

# Create geometry column
df['geometry'] = df.apply(lambda row: Polygon((p["lon"], p["lat"]) for p in row.geometry), axis=1)

# Convert to GeoDataFrame
lock_geometries = gpd.GeoDataFrame(df, geometry="geometry", crs="EPSG:4326")

# We get two objects that are similar
lock_geometries.head()
lock_geometries['name'] = ['Westsluis','Oostsluis','Nieuwe Sluis']

For plotting purposes, we define a subgraph:

In [ ]:
sub_FG = FG.subgraph(find_nodes_in_a_polygon(FG, area_of_interest))

In [ ]:
m = plot_graph_folium(sub_FG, longitude=3.825, latitude=51.325, zoom_start=13)
for index, lock_info in lock_geometries.iterrows():
    visualize_geometry_polygon_in_folium_plot(m, lock_info.geometry, label = lock_info['name'])
m

We see that there are nodes in the lock chamber, which is not allowed. We will identify them and try to remove them:

In [ ]:
nodes_within_the_lock_chamber = []
for lock_geometry in lock_geometries.geometry:
    nodes_within_the_lock_chamber.extend(find_nodes_in_a_polygon(FG, lock_geometry))
nodes_within_the_lock_chamber

In [ ]:
modified_FG = merge_two_consecutive_edges_based_on_shared_node(FG, nodes_within_the_lock_chamber)

We notice that removing the notes will create two edges between the same node-pair: <b>NL_J1754</b> and <b>NL_J1749</b>. This is not allowed for the default EURIS-network, which is of type <i>nx.Graph</i>. Instead, we need to create a <i>nx.MultiGraph</i>-type.

In [ ]:
multi_FG = nx.MultiGraph(FG)
modified_FG = merge_two_consecutive_edges_based_on_shared_node(multi_FG, nodes_within_the_lock_chamber)

We can visually check the graph:

In [ ]:
sub_FG = modified_FG.subgraph(find_nodes_in_a_polygon(modified_FG, area_of_interest))

In [ ]:
m = plot_graph_folium(sub_FG, longitude=3.825, latitude=51.325, zoom_start=13)
for index, lock_info in lock_geometries.iterrows():
    visualize_geometry_polygon_in_folium_plot(m, lock_info.geometry)
m

In [ ]:
env.graph = modified_FG

#### 1+ Adding hydrodynamics

In [ ]:
hydrodynamic_data = xr.open_dataset(Path.cwd() / 'data/Locking/hydrodynamic_data_Terneuzen.nc')

In [ ]:
location_to_node = {}
for station in hydrodynamic_data.STATION.values:
    station_data = hydrodynamic_data.sel({'STATION':station})
    point = Point(station_data.LON.values, station_data.LAT.values)
    crs_from = str(station_data.EPSG.values)
    crs_to = 'EPSG:4326'
    transformer = Transformer.from_crs(crs_from, crs_to, always_xy = True)
    point = Point(transformer.transform(point.x,point.y))
    node = get_closest_node_to_geometry(modified_FG, point)
    location_to_node[str(station)] = node

In [ ]:
destinations_canalside = []
for destination in ['NL_J1749', 'NL_J1751']:
    if destination not in destinations_canalside:
        destinations_canalside.append(destination)

In [ ]:
hydrodynamic_data = hydrodynamic_data.assign_coords(STATION=list(location_to_node.values()))

In [ ]:
extrapolated_nodes_canalside = []
origin_canalside = location_to_node['Terneuzen, Sluiskilbrug']
for destination in destinations_canalside:
    route = nx.dijkstra_path(modified_FG, origin_canalside, destination)
    for node in route:
        if node not in extrapolated_nodes_canalside:
            extrapolated_nodes_canalside.append(node)
extrapolated_nodes = {origin_canalside:extrapolated_nodes_canalside}

origin_seaside_1 = location_to_node['Terneuzen, westsluis, buiten']
destination_seaside_1 = 'NL_J0561'
route = nx.dijkstra_path(modified_FG, origin_seaside_1, destination_seaside_1)
extrapolated_nodes[origin_seaside_1] = route

origin_seaside_2 = location_to_node['Terneuzen']
destination_seaside_2 = 'NL_J0560'
route = nx.dijkstra_path(modified_FG, origin_seaside_2, destination_seaside_2)
extrapolated_nodes[origin_seaside_2] = route

In [ ]:
for original_node, extrapolation_nodes in extrapolated_nodes.items():
    copied = hydrodynamic_data.sel({'STATION':original_node})
    for extra_node in extrapolation_nodes:
        if extra_node in hydrodynamic_data.STATION.values:
            continue
        # Assign new coordinate value
        copied = copied.assign_coords(STATION=extra_node)
        
        # Concatenate along STATION dimension
        hydrodynamic_data = xr.concat(
            [hydrodynamic_data, copied],
            dim="STATION"
        )

In [ ]:
HydrodynamicData(env=env, hydrodynamic_data = hydrodynamic_data);

#### 1+ Infrastructure

In [ ]:
lock_geometries['length'] = [290., 280., 427.] #Based on RWS-website
lock_geometries['width'] = [40., 24., 55.] #Based on RWS-website
lock_geometries['depth'] = [13.5, 6.5, 16.4] #Based on RWS-website
lock_geometries['gate_opening_time'] = [150, 90, 180] #Best-guess values
lock_geometries['gate_closing_time'] = lock_geometries['gate_opening_time'] #Best-guess values
lock_geometries['opening_depth'] = lock_geometries['depth']*0.9 #Best-guess values
lock_geometries['opening_area'] = [18,12,32] #Best-guess values

In [ ]:
lock_chambers = {}
for _, lock_info in lock_geometries.iterrows():
    lock_chambers[lock_info['name']] = IsLockChamber(env=env,
                                                     name=lock_info['name'],
                                                     geometry = lock_info.geometry,
                                                     lock_length = lock_info['length'],
                                                     lock_width = lock_info['width'],
                                                     lock_depth = lock_info['depth'],
                                                     gate_opening_time = lock_info['gate_opening_time'],
                                                     gate_closing_time = lock_info['gate_closing_time'],
                                                     opening_depth = lock_info['opening_depth'],
                                                     opening_area = lock_info['opening_area'],
                                                     crs_m='EPSG:28992') #Amersfoort / RD New reference system

In [ ]:
# The minimum required input for a lock complex are waiting areas at both sides of the lock
waiting_areas = {}
index = 0
waiting_area_names = ['A', 'B', 'C', 'D', 'E', 'F']
waiting_area_distances = [650, 700, 420, 300, 450, 800] #distances to the lock gate have been made using the ruler-tool in Google Earth
for lock_chamber_name, lock_chamber in lock_chambers.items():
    waiting_area_name = waiting_area_names[index]
    waiting_areas[waiting_area_name] = IsLockWaitingArea(env=env,
                                                         name = f"Waiting area {waiting_area_name}",
                                                         lock_chamber = lock_chamber,
                                                         distance_from_lock_gate_A = waiting_area_distances[index])
    index += 1
    waiting_area_name = waiting_area_names[index]
    waiting_areas[waiting_area_name] = IsLockWaitingArea(env=env,
                                                         name = f"Waiting area {waiting_area_name}",
                                                         lock_chamber = lock_chamber,
                                                         distance_from_lock_gate_B = waiting_area_distances[index])
    index += 1

In [ ]:
lock_complex = IsLockComplex(lock_chambers = lock_chambers,
                             waiting_areas = waiting_areas,
                             registration_nodes = ['NL_J1986','NL_J0561'],
                             env=env,
                             name = 'Lock complex',)

We can visually inspect the lock complex:

In [ ]:
spatially_visualize_lock_complex(lock_complex)

#### 2. Create agents
We create agents according to earlier notebooks.

In [ ]:
Vessel = create_object(
    "Vessel",
    (
        Identifiable,               # allows to give the object a name and a random ID,
        VesselProperties,           # allows vessel to have dimensions, namely a length (L), width (B), and draught (T)      
        Movable,                    # allows the object to move, with a fixed speed, while logging this activity
        LockComplexTraversable,     # allows to interact with a lock
        ConsumesEnergy,             # allows to calculate energy consumption
        ExtraMetadata,              # allow additional information, such as an arrival time (required for passing a lock)
        HasMultiDiGraph,            # allow to operate on a graph that can include parallel edges from and to the same nodes
        HasOutput,                  # allow additional output to be stored
        
    ),
)

In [ ]:
def mission(env, vessel):
    """
    Method that defines the mission of the vessel.
    
    In this case: 
        keep moving along the path until its end point is reached
    """
    while True:
        yield from vessel.move()
        
        if vessel.geometry == nx.get_node_attributes(env.graph, "geometry")[vessel.route[-1]]:
            break

In [ ]:
upstream_vessels = generate_vessels_from_distribution(env=env,
                                                      VesselClass = Vessel,
                                                      vessel_parameters = {
                                                          "v":4, 
                                                          "L":100, 
                                                          "B":20, 
                                                          "T":5, 
                                                          "type":'tanker',
                                                          "safety_margin": 0.2,       
                                                          "h_squat": True,           
                                                          "P_installed": 1750.0, 
                                                          "P_tot_given": None, 
                                                          "bulbous_bow": False, 
                                                          "karpov_correction": True, 
                                                          "P_hotel_perc": 0.05, 
                                                          "P_hotel": None,
                                                          "x": 2,
                                                          "L_w": 3.0 ,
                                                          "C_B": 0.85,
                                                          "C_year": 1990,
                                                      },
                                                      mean_arrival_rate=30.,
                                                      number_of_vessels=10,
                                                      start_node = 'NL_J0561',
                                                      end_node = 'NL_J1986',
                                                      seed = 123)

downstream_vessels = generate_vessels_from_distribution(env=env,
                                                        VesselClass = Vessel,
                                                        vessel_parameters = {
                                                            "v":4, 
                                                            "L":100, 
                                                            "B":20, 
                                                            "T":5, 
                                                            "type":'tanker',
                                                            "safety_margin": 0.2,       
                                                            "h_squat": True,           
                                                            "P_installed": 1750.0, 
                                                            "P_tot_given": None, 
                                                            "bulbous_bow": False, 
                                                            "karpov_correction": True, 
                                                            "P_hotel_perc": 0.05, 
                                                            "P_hotel": None,
                                                            "x": 2,
                                                            "L_w": 3.0 ,
                                                            "C_B": 0.85,
                                                            "C_year": 1990,
                                                        },
                                                        mean_arrival_rate=30.,
                                                        number_of_vessels=10,
                                                        start_node = 'NL_J1986',
                                                        end_node = 'NL_J0561',
                                                        seed = 456)

vessels = upstream_vessels + downstream_vessels

for vessel in vessels:
    env.process(mission(env, vessel))

#### 3. Run simulation
We run the simulation

In [ ]:
env.run()

#### 4. Inspect output
##### Traffic dynamics
We first inspect the time-distance diagrams of each lock chamber:

In [ ]:
def cm_to_pixels(cm):
    return cm * 37.8 # Set figure height to 10 cmfig.update_layout(height=cm_to_pixels(10))

# We can plot the time-distance diagram
fig = lock_chambers['Westsluis'].plot(xlimmin = -6050, 
                                      xlimmax = 6050,
                                      ylimmin = pd.Timestamp('2026-01-01 00:00:00'),
                                      ylimmax = pd.Timestamp('2026-01-01 06:00:00'),
                                      method='Plotly',
                                      boundary_nodes = ['NL_J1986','NL_J0561'])

fig.update_layout(height=cm_to_pixels(20))

In [ ]:
# We can plot the time-distance diagram
fig = lock_chambers['Oostsluis'].plot(xlimmin = -6050, 
                                      xlimmax = 6050,
                                      ylimmin = pd.Timestamp('2026-01-01 00:00:00'),
                                      ylimmax = pd.Timestamp('2026-01-01 06:00:00'),
                                      method='Plotly',
                                      boundary_nodes = ['NL_J1986','NL_J0561'])

fig.update_layout(height=cm_to_pixels(20))

In [ ]:
# We can plot the time-distance diagram
fig = lock_chambers['Nieuwe Sluis'].plot(xlimmin = -6050, 
                                      xlimmax = 6050,
                                      ylimmin = pd.Timestamp('2026-01-01 00:00:00'),
                                      ylimmax = pd.Timestamp('2026-01-01 06:00:00'),
                                      method='Plotly',
                                      boundary_nodes = ['NL_J1986','NL_J0561'])

fig.update_layout(height=cm_to_pixels(20))

##### Performance

In [ ]:
performances_lock_chambers = {}
for lock_chamber_name, lock_chamber in lock_chambers.items():
    performances_lock_chambers[lock_chamber_name] = lock_chamber.get_performance()

In [ ]:
performances_lock_chambers['Nieuwe Sluis']['Saltwater intrusion [kg]'],
performances_lock_chambers['Oostsluis']['Saltwater intrusion [kg]'],
performances_lock_chambers['Westsluis']['Saltwater intrusion [kg]']